# CS485 Homework 1: Exploratory Data Analysis

## Part 0: Introduction & Data Setup

**Dataset Chosen:** Multimodal Universe - SDSS Galaxy Catalog

**Selected Modalities:**
For this analysis, two modalities are integrated:
1. **Photometric Data:** 5 broad-band magnitudes (u, g, r, i, z) capturing overall brightness.
2. **Spectral Data:** 5 spectral line equivalent widths (H-alpha, OIII etc..) providing chemical composition.

**Data Matrix Definition ($X$):**
The dataset is represented as a data matrix $X \in \mathbb{R}^{n \times d}$. 
* **$n$ (Samples):** Represents 1000 distinct galaxies.
* **$d$ (Features):** 10 combined features (5 photometric + 5 spectral).
* Therefore, $X \in \mathbb{R}^{1000 \times 10}$.

**Target Vector ($y$):**
The target vector $y \in \mathbb{R}^{n}$ represents the spectroscopic redshift of each galaxy.

In [ ]:
#fundamental library imports
import numpy as np
import pandas as pd

#plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

#tools for scaling and similarity
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

#matplotlib config for inline plots
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid') #specify style

from datasets import load_dataset

#download the SDSS dataset and convert to pandas dataframe
dataset = load_dataset('MultimodalUniverse/sdss', split='train[:1000]')
df = dataset.to_pandas()

#save the dataframe to a local CSV
csv_filename = 'sdss_data.csv'
df.to_csv(csv_filename, index=False)

print(f"Dataset shape: {df.shape}")
print(f"Saved locally as: {csv_filename}")

## Part 1 — Exploratory Data Analysis

### 3.1 Data Encoding & Representation
To begin with the analysis, we need to load the dataset, pinpoint the dataset column names, construct the feature matrix $X$ and extract the target vector $y$.

In [ ]:
#load the dataset downloaded from github
df = pd.read_csv('sdss_data.csv')

#print all column names in dataset
print(df.columns.tolist())
#look at the first 3 rows to see what the data actually is
display(df.head(3))

#the csv has columns for magnitudes, spectral lines and redshift
#drop the target column to create the feature matrix X
X_df = df.drop(columns = ['Z'])
y = df['Z'].values

#convert to numpy array for matrix operations
X = X_df.values

print(f"Shape of Data Matrix X: {X.shape}")
print(f"Shape of Target Vector y: {y.shape}")

**Observations and Comments (3.1):**
* **Encoding Choices:** All features are continuous numerical variables (64-bit floats). No categorical encoding required.
* **Missing Values:** For this dataset, any missing spectral lines (NaNs) would be infered using the column mean to preserve matrix dimensions without losing galaxy samples.
* **Scales:** The photometric magnitudes (values around 15-20) and spectral widths (values around 0-5) are on different scales. Standardizing is necessary before utilizing gradient descent.

### 3.2 Similarity & Inner Products
Now, we need to compute the pairwise cosine similarity for a representative subset of 50 galaxies to understand how vectors relate in the 10-dimensional feature space.

In [ ]:
#Select subset of 50 objects
X_subset = X[:50]

#compute cosine similarity and standardise first so features with larger magnitudes dont dominate the angle
scaler = StandardScaler()
X_subset_scaled = scaler.fit_transform(X_subset)
similarity_matrix = cosine_similarity(X_subset_scaled)

#plot the heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(similarity_matrix, cmap='viridis', annot=False)
plt.title('Cosine Similarity Heatmap (Subset of 50 Galaxies)')
plt.xlabel('Galaxy Index')
plt.ylabel('Galaxy Index')
plt.show()